<a href="https://colab.research.google.com/github/Ronniee7/API-Key-for-Model/blob/main/CNN-based%20Plant%20Disease%20Detection%20System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project: PlantDoc CNN
Intelligent Plant Disease Classification Using Convolutional Neural Networks
CNN
TensorFlow
Computer Vision


Dataset: Beans leaf disease dataset (3 classes)

Classes: Healthy, angular leaf spot, and bean rust.

What you will implement
1. Load and explore a real image dataset.
2. Perform image preprocessing and normalization.
3. Apply data augmentation.
4. Design a CNN architecture from scratch.
5. Train the CNN using Adam and categorical cross-entropy.
6. Evaluate accuracy, precision, recall, F1-score and confusion matrix.
7. Visualize training and validation performance.
8. Use Grad-CAM to visualize which parts of a leaf influenced the prediction.
9. Test the trained model on an individual image.

In [5]:
!pip -q install tensorflow-datasets seaborn scikit-learn
!pip install -q importlib_resources tensorflow-datasets

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.21.0
GPU available: []


##2 — Load the dataset

We use the Beans dataset, containing real photographs of bean leaves.

In [6]:
(ds_train, ds_val, ds_test), info = tfds.load(
    'beans',
    split=['train', 'validation', 'test'],
    as_supervised=True,
    with_info=True
)

class_names = info.features['label'].names
NUM_CLASSES = len(class_names)

print("Classes:", class_names)
print("Training images:", info.splits['train'].num_examples)
print("Validation images:", info.splits['validation'].num_examples)
print("Testing images:", info.splits['test'].num_examples)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

DownloadError: Failed to get url https://storage.googleapis.com/ibeans/train.zip. HTTP code: 403.

##3 — Explore the dataset
Visualize some actual training images.

In [4]:
plt.figure(figsize=(12, 8))

for images, labels in ds_train.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i])
        plt.title(class_names[labels[i].numpy()])
        plt.axis('off')

plt.suptitle("Sample Bean Leaf Images", fontsize=16)
plt.tight_layout()
plt.show()

NameError: name 'ds_train' is not defined

<Figure size 1200x800 with 0 Axes>

##4 — Preprocess the images
CNNs require consistent image dimensions.

We resize every image to 224 × 224 pixels.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds = (
    ds_train
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .shuffle(500)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    ds_val
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    ds_test
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

##5 — Data augmentation
Data augmentation helps the CNN generalize to different orientations and appearances of leaves.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15)
], name="data_augmentation")

##6 — Build the CNN architecture
This is the main deep learning component.

Our CNN contains:
- Convolutional layers for feature extraction.
- ReLU activation for non-linearity.
- Max pooling for dimensionality reduction.
- Batch normalization for training stability.
- Dropout for regularization.
- Dense layers for classification.
- Softmax for multiclass prediction.

In [ ]:

    tf.keras.layers.Conv2D(
        32, (3, 3), padding='same',
        activation='relu'
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(
        64, (3, 3), padding='same',
        activation='relu'
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(
        128, (3, 3), padding='same',
        activation='relu'
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(
        256, (3, 3), padding='same',
        activation='relu',
        name='last_conv'
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

model.summary()

##7 — Compile the model
We use Adam as our optimizer and sparse categorical cross-entropy as our loss function.

Why sparse categorical cross-entropy? Because our labels are integers (0, 1, 2), rather than one-hot encoded vectors.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']
)

##8 — Train the CNN
We use early stopping to prevent unnecessary training and restore the best model weights.

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[early_stopping, reduce_lr]

##9 — Visualize training performance
This is important when analyzing whether your model is overfitting or underfitting.

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

##10 — Evaluate on unseen test data
Do not use the test set to tune the model. It should be reserved for final evaluation.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)
print("Test Accuracy (%):", round(test_accuracy * 100, 2))

##11 — Confusion matrix and classification report
Accuracy alone is not enough. We need to understand which disease classes the model confuses.

In [ ]:
y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("CNN Confusion Matrix")
plt.show()

##12 — Test individual predictions
Let's visualize the model's predictions and confidence scores.

In [ ]:
plt.figure(figsize=(14, 10))

for images, labels in test_ds.take(1):
    predictions = model.predict(images, verbose=0)

    for i in range(9):
        predicted_class = np.argmax(predictions[i])
        confidence = np.max(predictions[i]) * 100

        plt.subplot(3, 3, i + 1)

        plt.imshow(images[i].numpy())

        actual = class_names[labels[i].numpy()]
        predicted = class_names[predicted_class]

        plt.title(
            f"Actual: {actual}\n"
            f"Predicted: {predicted}\n"
            f"Confidence: {confidence:.1f}%"
        )

        plt.axis('off')

plt.tight_layout()
plt.show()

##13 — Explain predictions using Grad-CAM
This is one of the most interesting parts of the project.

Grad-CAM helps us visualize the regions of an image that influenced the CNN's prediction.

In [ ]:
def make_gradcam_heatmap(image, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(
            tf.expand_dims(image, axis=0),
            training=False
        )

        class_index = tf.argmax(predictions[0])
        loss = predictions[:, class_index]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(
        grads,
        axis=(0, 1, 2)
    )

    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(
        conv_outputs * pooled_grads,
        axis=-1
    )

    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + 1e-8

    return heatmap.numpy()


for images, labels in test_ds.take(1):

    image = images[0]

    heatmap = make_gradcam_heatmap(
        image,
        model,
        'last_conv'
    )

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title("Original Image")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(image)
    plt.imshow(
        heatmap,
        cmap='jet',
        alpha=0.4,
        extent=(0, IMG_SIZE, IMG_SIZE, 0)
    )
    plt.title("Grad-CAM Visualization")
    plt.axis('off')

    plt.show()

##14 — Save trained model

In [ ]:
model.save("plant_disease_cnn.keras")

print("Model saved successfully!")

In [ ]:
download the model

In [ ]:
from google.colab import files

files.download("plant_disease_cnn.keras")

##15 — Predict a completely new image
This allows you to upload a bean leaf photograph from your computer and classify it.

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()

for filename in uploaded.keys():

    image = Image.open(filename).convert("RGB")
    image = image.resize((IMG_SIZE, IMG_SIZE))

    image_array = np.array(image) / 255.0
    image_array = np.expand_dims(image_array, axis=0)

    prediction = model.predict(image_array, verbose=0)

    predicted_class = np.argmax(prediction)
    confidence = np.max(prediction) * 100

    plt.imshow(image)
    plt.axis('off')
    plt.title(
        f"Prediction: {class_names[predicted_class]}\n"
        f"Confidence: {confidence:.2f}%"
    )
    plt.show()